## Imports

In [1]:
from desafio_nava.utils.spark_config import init_spark
from pyspark.sql import functions as F
from datetime import date
import os
import shutil

## Start Spark Session

In [2]:
spark = init_spark(
        app_name="DesafioNavaApp",
        project_name="desafio_nava"
    )


✅ Spark 3.5.7 iniciado com Hive local persistente!
📦 Projeto:   desafio_nava
📁 Warehouse: D:/Projetos/desafio_nava/delta_lake/spark-warehouse
📁 Metastore: D:/Projetos/desafio_nava/delta_lake/metastore_db



## Variables

In [3]:
# Define caminhos locais onde serão armazenadas as tabelas Delta
BASE_BRONZE_PATH = "D:/Projetos/desafio_nava/data/bronze"
BASE_SILVER_PATH = "D:/Projetos/desafio_nava/data/silver"
BASE_GOLD_PATH = "D:/Projetos/desafio_nava/data/gold"

# Caminho dos arquivos CSV brutos
RAW_CSV_GLOB = "D:/Projetos/desafio_nava/data/raw/pda-024-icb-*.csv"

# Bronze
DELTA_PATH_RAW_PDA_BENEFICIARIO = f"{BASE_BRONZE_PATH}/raw_pda_beneficiario"

# Silver
DELTA_PATH_STG_PDA_BENEFICIARIO = f"{BASE_SILVER_PATH}/stg_pda_beneficiario"

# Dimensões
DELTA_PATH_DIM_ATIVO   = f"{BASE_GOLD_PATH}/dim_ativo_financeiro"
DELTA_PATH_DIM_TEMPO   = f"{BASE_GOLD_PATH}/dim_tempo"
DELTA_PATH_DIM_CLIENTE = f"{BASE_GOLD_PATH}/dim_cliente"

# Fato
DELTA_PATH_FATO_COTACAO = f"{BASE_GOLD_PATH}/fato_cotacao"


## Create Schemas

In [4]:
 # BRONZE - Camada de dados RAW
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS bronze
    LOCATION '{BASE_BRONZE_PATH}'
    COMMENT 'Camada Bronze - Dados brutos sem transformação. 
            Dados exatamente como chegam dos arquivos CSV.
            Todos os campos em STRING para preservar dados originais.'
    WITH DBPROPERTIES (
        'layer' = 'bronze',
        'data_quality' = 'raw',
        'created_date' = '2026-03-26',
        'sla' = 'near_realtime',
        'retention_days' = '30',
        'pii_data' = 'yes'
    )
""")

print("✅ Schema bronze criada com sucesso!")

✅ Schema bronze criada com sucesso!


In [5]:
# Schema Silver
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS silver
    LOCATION '{BASE_SILVER_PATH}'
    COMMENT 'Camada Silver - Dados limpos e validados'
    WITH DBPROPERTIES (
        'layer' = 'silver',
        'data_quality' = 'cleaned_and_validated',
        'created_date' = '2026-03-26',
        'sla' = 'daily',
        'retention_days' = '365',
        'pii_data' = 'masked'
    )
""")

print("✅ Schema silver criada com sucesso!")

✅ Schema silver criada com sucesso!


In [6]:
# Schema Gold
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS gold
    LOCATION '{BASE_GOLD_PATH}'
    COMMENT 'Camada Gold - Dados agregados e métricas'
    WITH DBPROPERTIES (
        'layer' = 'gold',
        'data_quality' = 'curated',
        'created_date' = '2026-03-26',
        'sla' = 'daily',
        'retention_days' = '2555',
        'pii_data' = 'no'
    )
""")

print("✅ Schema gold criada com sucesso!")

✅ Schema gold criada com sucesso!


## Create Table Bronze

In [7]:
# Remove o diretório da tabela Delta se ele existir
if os.path.exists(DELTA_PATH_RAW_PDA_BENEFICIARIO):
    shutil.rmtree(DELTA_PATH_RAW_PDA_BENEFICIARIO)
    print(f"✅ Diretório removido: {DELTA_PATH_RAW_PDA_BENEFICIARIO}")

# Remove a tabela do metastore se ela existir
spark.sql("DROP TABLE IF EXISTS bronze.raw_pda_beneficiario")
print("✅ Tabela removida do metastore")


spark.sql(f"""
CREATE TABLE IF NOT EXISTS bronze.raw_pda_beneficiario (
    ID_CMPT_MOVEL            STRING               COMMENT 'Competência de referência no formato YYYY-MM',
    CD_OPERADORA             STRING               COMMENT 'Código da operadora na ANS',
    NM_RAZAO_SOCIAL          STRING               COMMENT 'Razão social da operadora',
    NR_CNPJ                  STRING               COMMENT 'CNPJ da operadora (com zeros à esquerda)',
    MODALIDADE_OPERADORA     STRING               COMMENT 'Modalidade jurídica da operadora',
    SG_UF                    STRING               COMMENT 'Sigla do estado (UF)',
    CD_MUNICIPIO             STRING               COMMENT 'Código IBGE do município',
    NM_MUNICIPIO             STRING               COMMENT 'Nome do município',
    TP_SEXO                  STRING               COMMENT 'Sexo do beneficiário (M/F)',
    DE_FAIXA_ETARIA          STRING               COMMENT 'Faixa etária do beneficiário',
    DE_FAIXA_ETARIA_REAJ     STRING               COMMENT 'Faixa etária para fins de reajuste',
    CD_PLANO                 STRING               COMMENT 'Código do plano registrado na ANS',
    TP_VIGENCIA_PLANO        STRING               COMMENT 'Tipo de vigência do plano',
    DE_CONTRATACAO_PLANO     STRING               COMMENT 'Tipo de contratação (Individual/Coletivo)',
    DE_SEGMENTACAO_PLANO     STRING               COMMENT 'Segmentação assistencial do plano',
    DE_ABRG_GEOGRAFICA_PLANO STRING               COMMENT 'Abrangência geográfica do plano',
    COBERTURA_ASSIST_PLAN    STRING               COMMENT 'Tipo de cobertura assistencial',
    TIPO_VINCULO             STRING               COMMENT 'Vínculo do beneficiário (Titular/Dependente)',
    QT_BENEFICIARIO_ATIVO    STRING               COMMENT 'Quantidade de beneficiários ativos',
    QT_BENEFICIARIO_ADERIDO  STRING               COMMENT 'Quantidade de beneficiários aderidos no período',
    QT_BENEFICIARIO_CANCELADO STRING              COMMENT 'Quantidade de beneficiários cancelados no período',
    DT_CARGA                 STRING               COMMENT 'Data em que o arquivo foi carregado',
    CRIADO_EM                TIMESTAMP            COMMENT 'Timestamp de criação do registro'
)
USING DELTA
PARTITIONED BY (SG_UF, ID_CMPT_MOVEL)
LOCATION '{DELTA_PATH_RAW_PDA_BENEFICIARIO}'
COMMENT 'Dados brutos de beneficiários de planos de saúde — fonte ANS/PDA. Sem transformações aplicadas.'
"""
)

print("✅ Tabela raw_pda_beneficiario criada com sucesso!")


✅ Diretório removido: D:/Projetos/desafio_nava/data/bronze/raw_pda_beneficiario
✅ Tabela removida do metastore
✅ Tabela raw_pda_beneficiario criada com sucesso!


## Create Table Silver

In [8]:
# Remove o diretório da tabela Delta se ele existir
if os.path.exists(DELTA_PATH_STG_PDA_BENEFICIARIO):
    shutil.rmtree(DELTA_PATH_STG_PDA_BENEFICIARIO)
    print(f"✅ Diretório removido: {DELTA_PATH_STG_PDA_BENEFICIARIO}")

# Remove a tabela do metastore se ela existir
spark.sql("DROP TABLE IF EXISTS silver.stg_pda_beneficiario")
print("✅ Tabela removida do metastore")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS silver.stg_pda_beneficiario (
    ID_CMPT_MOVEL            STRING               COMMENT 'Competência de referência no formato YYYY-MM',
    CD_OPERADORA             STRING               COMMENT 'Código da operadora na ANS (preservado como string)',
    NM_RAZAO_SOCIAL          STRING               COMMENT 'Razão social da operadora',
    NR_CNPJ                  STRING               COMMENT 'CNPJ da operadora (preservado como string para zeros à esquerda)',
    MODALIDADE_OPERADORA     STRING               COMMENT 'Modalidade jurídica da operadora',
    SG_UF                    STRING               COMMENT 'Sigla do estado (UF)',
    CD_MUNICIPIO             STRING               COMMENT 'Código IBGE do município (preservado como string)',
    NM_MUNICIPIO             STRING               COMMENT 'Nome do município',
    TP_SEXO                  STRING               COMMENT 'Sexo do beneficiário (M/F)',
    DE_FAIXA_ETARIA          STRING               COMMENT 'Faixa etária do beneficiário',
    DE_FAIXA_ETARIA_REAJ     STRING               COMMENT 'Faixa etária para fins de reajuste',
    CD_PLANO                 STRING               COMMENT 'Código do plano registrado na ANS (preservado como string)',
    TP_VIGENCIA_PLANO        STRING               COMMENT 'Tipo de vigência do plano (ex: P = Posterior à Lei)',
    DE_CONTRATACAO_PLANO     STRING               COMMENT 'Tipo de contratação (Individual/Coletivo por Adesão etc.)',
    DE_SEGMENTACAO_PLANO     STRING               COMMENT 'Segmentação assistencial do plano',
    DE_ABRG_GEOGRAFICA_PLANO STRING               COMMENT 'Abrangência geográfica do plano',
    COBERTURA_ASSIST_PLAN    STRING               COMMENT 'Tipo de cobertura assistencial',
    TIPO_VINCULO             STRING               COMMENT 'Vínculo do beneficiário (Titular/Dependente)',
    QT_BENEFICIARIO_ATIVO    INT                  COMMENT 'Quantidade de beneficiários ativos',
    QT_BENEFICIARIO_ADERIDO  INT                  COMMENT 'Quantidade de beneficiários aderidos no período',
    QT_BENEFICIARIO_CANCELADO INT                 COMMENT 'Quantidade de beneficiários cancelados no período',
    DT_CARGA                 DATE                 COMMENT 'Data em que o arquivo foi carregado (formato YYYY-MM-DD)',
    CRIADO_EM                TIMESTAMP            COMMENT 'Timestamp de criação do registro na camada silver'
)
USING DELTA
PARTITIONED BY (SG_UF, ID_CMPT_MOVEL)
LOCATION '{DELTA_PATH_STG_PDA_BENEFICIARIO}'
COMMENT 'Dados de beneficiários de planos de saúde com tipos corrigidos — camada silver.'
""")

print("✅ Tabela stg_pda_beneficiario criada com sucesso!")


✅ Tabela removida do metastore
✅ Tabela stg_pda_beneficiario criada com sucesso!


## Create Dimension Tables - Gold

## Create Fact Table - Gold

## Stop Spark Session

In [9]:
# Encerra a SparkSession
spark.stop()